# Final Project

## Part 0 Project Proposal
Before we do the project, you will need to create a repository
1. Create the project remotely on GitHub.  You will want to include a README
2. Install GitHub Desktop.
3. From the remote repo, clone down to your local and use github desktop to create it.
4. Copy the notebook into your local repo
5. Commit and push

***Invite me and the GA to your repo***

As you make changes locally, commit and push them. 

To complete the proposal. You will record the following in your README

1. The kind of data you want to find
2. The question/s you want to answer with the data.
3. URLs for the site/s that have the data you want.

***Note: You must aquire the data via an API or Web scraping.  Downloading a file will not get points.***

All code for your project will be recorded in this note book. Create extra code cells as needed.


# Part 1: Data aquisition
    1. Get raw data and put it into files. If needed, gather a representative amount of data.  Then append additional data as available.
    


In [2]:
# Part 1: Put code here

import requests
import pandas as pd
import datetime
import time
import os

# ─────────────────────────────────────────────
# 1. Configuration
# ─────────────────────────────────────────────
SUBREDDITS   = ['fitness', 'weightlifting']
POST_LIMIT   = 500        # 100 posts per subreddit. Increase to 500+ overtime
COMMENT_LIMIT = 10     
OUTPUT_DIR   = 'data'  

HEADERS = {'User-Agent': 'python:university_research_project:v1.0 (by /u/SuspiciousKiwi3427'}

os.makedirs(OUTPUT_DIR, exist_ok=True)



In [3]:
# ─────────────────────────────────────────────
# 2. Scraping functions
# ─────────────────────────────────────────────
def get_posts(subreddit_name, limit=100):
    """Fetch top posts from a subreddit using Reddit's public JSON endpoint."""
    posts = []
    after = None 

    print(f'Scraping r/{subreddit_name}...')

    while len(posts) < limit:
        url = f'https://www.reddit.com/r/{subreddit_name}/top.json?t=year&limit=25'
        if after:
            url += f'&after={after}'

        response = requests.get(url, headers=HEADERS)

        if response.status_code != 200:
            print(f'  Request failed with status {response.status_code}. Stopping.')
            break

        data = response.json()['data']
        children = data['children']

        if not children:
            break

        for child in children:
            p = child['data']
            created = datetime.datetime.utcfromtimestamp(p['created_utc'])
            posts.append({
                'id':           p['id'],
                'subreddit':    subreddit_name,
                'title':        p['title'],
                'body':         p.get('selftext', ''),
                'score':        p['score'],
                'num_comments': p['num_comments'],
                'created_utc':  p['created_utc'],
                'created_date': created.strftime('%Y-%m-%d'),
                'month':        created.month,
                'year':         created.year,
            })

        after = data['after']  # move to next page
        if not after:
            break

        time.sleep(2)

    print(f'  Collected {len(posts)} posts.')
    return pd.DataFrame(posts[:limit])


def get_comments(post_id, subreddit_name, limit=10):
    """Fetch top-level comments for a single post."""
    url = f'https://www.reddit.com/r/{subreddit_name}/comments/{post_id}.json?limit={limit}'
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return []

    comments = []
    try:
        comment_data = response.json()[1]['data']['children']
        for c in comment_data[:limit]:
            if c['kind'] == 't1':
                comments.append({
                    'post_id':    post_id,
                    'subreddit':  subreddit_name,
                    'comment_id': c['data']['id'],
                    'body':       c['data']['body'],
                    'score':      c['data']['score'],
                    'created_utc': c['data']['created_utc'],
                })
    except Exception as e:
        print(f'  Could not parse comments for post {post_id}: {e}')

    return comments

In [5]:
# ─────────────────────────────────────────────
# 3. Run scraper for both subreddits
# ─────────────────────────────────────────────
all_posts = []
all_comments = []

for sub in SUBREDDITS:
    posts_df = get_posts(sub, limit=POST_LIMIT)
    all_posts.append(posts_df)

    print(f'  Fetching comments for r/{sub}...')
    for post_id in posts_df['id']:
        comments = get_comments(post_id, sub, limit=COMMENT_LIMIT)
        all_comments.extend(comments)
        time.sleep(1)  # pause between comment requests

    print(f'  Done with r/{sub}.')

if all_posts:
    posts_combined    = pd.concat(all_posts, ignore_index=True)
    comments_combined = pd.DataFrame(all_comments)

    # CSV
    posts_combined.to_csv(f'{OUTPUT_DIR}/raw_posts.csv', index=False)
    comments_combined.to_csv(f'{OUTPUT_DIR}/raw_comments.csv', index=False)

    # Pickle
    posts_combined.to_pickle(f'{OUTPUT_DIR}/raw_posts.pkl')
    comments_combined.to_pickle(f'{OUTPUT_DIR}/raw_comments.pkl')
else:
    print("No data was collected at all. Check your connection or User-Agent.")

print(f'\nSaved {len(posts_combined)} posts → raw_posts.csv + raw_posts.pkl')
print(f'Saved {len(comments_combined)} comments → raw_comments.csv + raw_comments.pkl')
print(f'\nSubreddit breakdown:')
print(posts_combined['subreddit'].value_counts())
print(f'\nDate range: {posts_combined["created_date"].min()} to {posts_combined["created_date"].max()}')
posts_combined.head()

Scraping r/fitness...


C:\Users\S567597\AppData\Local\Temp\ipykernel_7816\1494212619.py:30: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  created = datetime.datetime.utcfromtimestamp(p['created_utc'])


  Collected 496 posts.
  Fetching comments for r/fitness...
  Done with r/fitness.
Scraping r/weightlifting...
  Collected 498 posts.
  Fetching comments for r/weightlifting...
  Done with r/weightlifting.

Saved 994 posts → raw_posts.csv + raw_posts.pkl
Saved 2114 comments → raw_comments.csv + raw_comments.pkl

Subreddit breakdown:
subreddit
weightlifting    498
fitness          496
Name: count, dtype: int64

Date range: 2025-04-16 to 2026-04-15


,id,subreddit,title,body,score,num_comments,created_utc,created_date,month,year
0,1kwo0r3,fitness,Tri-Annual Protein Megathread,**Welcome to the Tri-Annual Protein Megathread...,122,170,1.748354e+09,2025-05-27,5,2025
1,1mrnmc2,fitness,Gym Story Saturday,Hi! Welcome to your weekly thread where you ca...,107,94,1.755328e+09,2025-08-16,8,2025
2,1l5evlz,fitness,Gym Story Saturday,Hi! Welcome to your weekly thread where you ca...,98,118,1.749280e+09,2025-06-07,6,2025
3,1mv9kxu,fitness,"Rant Wednesday - August 20, 2025",Welcome to Rant Wednesday: It’s your time to l...,100,274,1.755680e+09,2025-08-20,8,2025
4,1m3q4x6,fitness,Gym Story Saturday,Hi! Welcome to your weekly thread where you ca...,96,127,1.752910e+09,2025-07-19,7,2025


## Part 2 Data processing

1. Process the raw data and store results in a file as needed.
2. Do analysis of the data
   

In [2]:
#Part 2: Put code here



## Part 3 Visualization

1. Create good graphs
2. Give narative conclusions explaining what is being demonstrated in the visualizations.

   

In [3]:
#Part 3: Put code here

